In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
print("Todos los imports funcionan correctamente")

Todos los imports funcionan correctamente


In [2]:
# 1) LEEMOS EL FICHERO EXCEL
RUTA_EXCEL = "C:/Users/chind/Desktop/PEOPLE ANALYTICS CEF UDIMA/Práctica Final_.xlsx"
HOJA = "Práctica Final"

df = pd.read_excel(RUTA_EXCEL, sheet_name=HOJA)
df = df.iloc[:, :11]

# Guardamos el nombre de la variable objetivo en una variable corta
BRADFORD = "Factor Bradford"

# Definimos las listas de variables que vamos a usar varias veces
VARIABLES_NUMERICAS = [
    "Edad",
    "Antigüedad",
    "Motivación (1-10)",
    "Satisfacción (1-10)",
    "Horas Extras",
    "Horas de Formación (h)",
    "Meses desde última promoción",
]

VARIABLES_CATEGORICAS = ["Departamento", "Tipo Contrato"]

print("=" * 70)
print("Datos cargados correctamente")
print("=" * 70)
print("Números de empleados:", len(df))
print("Columnas:", list(df.columns))
print()

Datos cargados correctamente
Números de empleados: 240
Columnas: ['ID Empleado', 'Departamento', 'Edad', 'Antigüedad', 'Tipo Contrato', 'Motivación (1-10)', 'Satisfacción (1-10)', 'Horas Extras', 'Horas de Formación (h)', 'Meses desde última promoción', 'Factor Bradford']



In [3]:
# 2) FUNCIÓN AUXILIAR: clasificar el absentismo en Bajo / Medio / Alto

def clasificar_absentismo(valor_bradford):
    "Devuelve el nivel de absentismo según el Factor Bradford"
    if valor_bradford < 50:
        return "Bajo"
    elif valor_bradford <= 200:
        return "Medio"
    else:
        return "Alto"
# Creamos una columna nueva con el nivel de cada empleado.
# .apply() aplica la función a cada valor de la columna.
df["Nivel Absentismo"] = df[BRADFORD].apply(clasificar_absentismo)

In [4]:
#  APARTADO 1: VARIABLES QUE INFLUYEN EN EL ABSENTISMO

print("=" * 70)
print(" 1. VARIABLES QUE INFLUYEN EN EL ABSENTISMO")
print("=" * 70)

# 1.A) Estadística descriptiva del absentismo
print("----- 1.A) Descripción general del Factor Bradford -----")
print("Media   :", round(df[BRADFORD].mean(), 2))
print("Mediana :", round(df[BRADFORD].median(), 2))
print("Mínimo  :", round(df[BRADFORD].min(), 2))
print("Máximo  :", round(df[BRADFORD].max(), 2))
print()
print("Reparto de la plantilla por nivel de absentismo:")
# value_counts() cuenta cuántos empleados hay en cada nivel
conteo_niveles = df["Nivel Absentismo"].value_counts()
for nivel in ["Bajo", "Medio", "Alto"]:
    cantidad = conteo_niveles.get(nivel, 0)
    porcentaje = cantidad / len(df) * 100
    print(f"   {nivel:6s}: {cantidad:3d} empleados ({porcentaje:.1f}%)")
print()
print(">> La media es muy superior a la mediana: unos pocos empleados con")
print("   absentismo muy alto disparan el promedio. La mayoría tiene valores bajos.")
print()

# 1.B) Correlaciones: ¿qué variable se relaciona más con el absentismo?
# La correlación de Pearson (r) va de -1 a +1:
#    r cercano a +1 -> cuando sube la variable, sube el absentismo
#    r cercano a -1 -> cuando sube la variable, BAJA el absentismo
#    r cercano a  0 -> no hay relación
# El "p-valor" indica si la relación es fiable: si p < 0.05 se considera
# estadísticamente significativa (no es fruto del azar).
print("----- 1.B) Correlación de cada variable con el absentismo -----")
print(f"{'Variable':<32}{'r (Pearson)':>12}{'p-valor':>10}  Significativa")
print("-" * 70)
 
# Guardamos los resultados para usarlos en un gráfico después
nombres_corr = []
valores_corr = []
 
for variable in VARIABLES_NUMERICAS:
    r, p = stats.pearsonr(df[variable], df[BRADFORD])
    es_significativa = "Sí" if p < 0.05 else "No"
    print(f"{variable:<32}{r:>+12.3f}{p:>10.3f}  {es_significativa}")
    nombres_corr.append(variable)
    valores_corr.append(r)
print()
print(">> Las variables con relación más fuerte (en valor absoluto) son:")
print("   Horas Extras (+), Motivación (-), Meses sin promoción (+) y Satisfacción (-).")
print(">> Edad y Antigüedad NO se relacionan con el absentismo: ni la veteranía")
print("   ni la edad explican las ausencias. Las causas son de gestión y clima laboral.")
print()

# 1.C) Absentismo medio por departamento
print("----- 1.C) Absentismo medio por DEPARTAMENTO -----")
# groupby agrupa por departamento y calcula media y mediana de cada grupo
tabla_dept = df.groupby("Departamento")[BRADFORD].agg(["mean", "median", "count"])
tabla_dept = tabla_dept.round(2).sort_values("mean", ascending=False)
print(tabla_dept.to_string())
print()
print(">> Ventas tiene la media más alta, pero su mediana es parecida al resto:")
print("   el problema lo causan unos pocos casos extremos, no todo el departamento.")
print()

# 1.D) Absentismo según el tipo de contrato (con test estadístico)
print("----- 1.D) Absentismo según TIPO DE CONTRATO -----")
tabla_contrato = df.groupby("Tipo Contrato")[BRADFORD].agg(["mean", "median", "count"])
print(tabla_contrato.round(2).to_string())
 
# Comparamos temporales vs indefinidos con un t-test:
# si p < 0.05, la diferencia es significativa; si no, puede ser casualidad.
grupo_temporal = df[df["Tipo Contrato"] == "Temporal"][BRADFORD]
grupo_indefinido = df[df["Tipo Contrato"] == "Indefinido"][BRADFORD]
t, p_contrato = stats.ttest_ind(grupo_temporal, grupo_indefinido)
print(f"\nTest t (Temporal vs Indefinido): p-valor = {p_contrato:.3f}")
if p_contrato < 0.05:
    print(">> La diferencia ES significativa.")
else:
    print(">> La diferencia NO es significativa: no podemos afirmar que el tipo")
    print("   de contrato, por sí solo, determine el absentismo.")
print()

# 1.E) Modelos: confirmar la importancia de cada variable
# Para meter las variables de texto (Departamento, Contrato) en un modelo,
# las convertimos en columnas de 0 y 1 con get_dummies (variables "dummy").
print("----- 1.E) Modelos explicativos (regresión y Random Forest) -----")
datos_modelo = pd.get_dummies(df, columns=VARIABLES_CATEGORICAS, drop_first=True)
 
# X = variables explicativas (las "causas"), y = lo que queremos explicar
columnas_X = VARIABLES_NUMERICAS + [
    c for c in datos_modelo.columns
    if c.startswith("Departamento_") or c.startswith("Tipo Contrato_")
]
X = datos_modelo[columnas_X].astype(float)
y = df[BRADFORD]
 
# --- Modelo 1: Regresión lineal ---
# Estandarizamos X para que los coeficientes sean comparables entre sí.
X_estandarizada = StandardScaler().fit_transform(X)
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_estandarizada, y)
r2_lineal = modelo_lineal.score(X_estandarizada, y)
print(f"R2 de la regresión lineal: {r2_lineal:.2f}")
print(f"   (las variables explican el {r2_lineal*100:.0f}% del absentismo)")
 
# --- Modelo 2: Random Forest (mide la importancia de cada variable) ---
modelo_bosque = RandomForestRegressor(n_estimators=300, random_state=42)
modelo_bosque.fit(X, y)
r2_bosque = modelo_bosque.score(X, y)
print(f"R2 del Random Forest: {r2_bosque:.2f}")
print()
print("Importancia de cada variable según el Random Forest (de mayor a menor):")
# Emparejamos cada variable con su importancia y las ordenamos
importancias = sorted(zip(columnas_X, modelo_bosque.feature_importances_),
                      key=lambda par: par[1], reverse=True)
for variable, importancia in importancias:
    print(f"   {variable:<32}{importancia*100:5.1f}%")
print()
print(">> Ambos modelos coinciden: motivación, satisfacción, horas extras y")
print("   meses sin promoción son los factores que más explican el absentismo.")
print()

 1. VARIABLES QUE INFLUYEN EN EL ABSENTISMO
----- 1.A) Descripción general del Factor Bradford -----
Media   : 111.63
Mediana : 20.59
Mínimo  : 0.0
Máximo  : 2288.33

Reparto de la plantilla por nivel de absentismo:
   Bajo  : 200 empleados (83.3%)
   Medio :  23 empleados (9.6%)
   Alto  :  17 empleados (7.1%)

>> La media es muy superior a la mediana: unos pocos empleados con
   absentismo muy alto disparan el promedio. La mayoría tiene valores bajos.

----- 1.B) Correlación de cada variable con el absentismo -----
Variable                         r (Pearson)   p-valor  Significativa
----------------------------------------------------------------------
Edad                                  +0.008     0.903  No
Antigüedad                            -0.010     0.883  No
Motivación (1-10)                     -0.806     0.000  Sí
Satisfacción (1-10)                   -0.741     0.000  Sí
Horas Extras                          +0.821     0.000  Sí
Horas de Formación (h)                -0.

In [5]:
#  APARTADO 2: PERFIL DEL TRABAJADOR ABSENTISTA
print("=" * 75)
print("  APARTADO 2: ¿EXISTE UN PERFIL DEL TRABAJADOR ABSENTISTA?")
print("=" * 75)
print()
 
# Comparamos el grupo de absentismo ALTO con el de absentismo BAJO
# para ver en qué se diferencian.
grupo_alto = df[df["Nivel Absentismo"] == "Alto"]
grupo_bajo = df[df["Nivel Absentismo"] == "Bajo"]
 
print("Comparación de las medias entre absentismo BAJO y ALTO:")
print(f"{'Variable':<32}{'Bajo':>10}{'Alto':>10}")
print("-" * 52)
for variable in VARIABLES_NUMERICAS:
    media_bajo = grupo_bajo[variable].mean()
    media_alto = grupo_alto[variable].mean()
    print(f"{variable:<32}{media_bajo:>10.1f}{media_alto:>10.1f}")
print()
 
# Vemos en qué departamentos se concentran los casos de absentismo alto
print("¿Dónde están los empleados de absentismo ALTO? (por departamento)")
reparto_alto = grupo_alto["Departamento"].value_counts(normalize=True) * 100
for departamento, porcentaje in reparto_alto.items():
    print(f"   {departamento:<22}{porcentaje:5.1f}%")
print()
 
print("----- CONCLUSIÓN DEL APARTADO 2 -----")
print("""
SÍ es posible definir un perfil del trabajador absentista.
 
PERFIL DE RIESGO (absentismo alto):
  - Motivación y satisfacción MUY BAJAS (en torno a 3 sobre 10).
  - MUCHAS horas extras acumuladas (señal de sobrecarga y agotamiento).
  - LLEVA MUCHO TIEMPO sin promocionar (estancamiento profesional).
  - Recibe MENOS formación que la media.
  - Se concentra sobre todo en el departamento de Ventas.
 
JUSTIFICACIÓN:
  - Los modelos explican entre el 81% y el 97% del absentismo, así que el
    perfil es estadísticamente sólido y no fruto del azar.
  - Las cuatro variables del perfil son justo las de mayor peso en los modelos.
 
CAUTELAS (importante para no sacar conclusiones erróneas):
  - Correlación no es causalidad: las horas extras pueden CAUSAR absentismo,
    o ambos pueden deberse a una mala organización del trabajo.
  - La edad, la antigüedad y el tipo de contrato NO sirven para perfilar.
    El perfil se basa en factores que la empresa PUEDE cambiar (motivación,
    carga de trabajo, desarrollo), no en etiquetar a las personas.
""")

  APARTADO 2: ¿EXISTE UN PERFIL DEL TRABAJADOR ABSENTISTA?

Comparación de las medias entre absentismo BAJO y ALTO:
Variable                              Bajo      Alto
----------------------------------------------------
Edad                                  44.4      44.5
Antigüedad                            10.3      10.4
Motivación (1-10)                      7.0       2.9
Satisfacción (1-10)                    7.5       2.8
Horas Extras                           4.7      31.5
Horas de Formación (h)                15.0       7.3
Meses desde última promoción          12.7      53.6

¿Dónde están los empleados de absentismo ALTO? (por departamento)
   Ventas                 64.7%
   Producción             23.5%
   Administración         11.8%

----- CONCLUSIÓN DEL APARTADO 2 -----

SÍ es posible definir un perfil del trabajador absentista.

PERFIL DE RIESGO (absentismo alto):
  - Motivación y satisfacción MUY BAJAS (en torno a 3 sobre 10).
  - MUCHAS horas extras acumuladas (señal d

In [6]:
#  APARTADO 3: ACCIONES PARA PREVENIR O REDUCIR EL ABSENTISMO
print("=" * 75)
print("  APARTADO 3: ACCIONES PARA REDUCIR EL ABSENTISMO")
print("=" * 75)
print("""
Las medidas se ordenan según el peso de cada variable en el análisis:
 
  1. MEJORAR MOTIVACIÓN Y SATISFACCIÓN (prioridad máxima).
     Encuestas de clima periódicas con planes de acción por departamento,
     reconocimiento del trabajo y revisión del encaje puesto-persona.
 
  2. CONTROLAR LA SOBRECARGA Y LAS HORAS EXTRAS.
     Poner límites a las horas extra, redimensionar plantillas en las áreas
     más tensionadas y vigilar el agotamiento como señal de alerta temprana.
 
  3. REACTIVAR LA PROMOCIÓN INTERNA.
     Diseñar planes de carrera y revisar los casos de empleados con más de
     tres años sin promocionar (muy ligados al absentismo alto).
 
  4. REFORZAR LA FORMACIÓN.
     Ampliar el plan formativo, sobre todo en quienes reciben menos horas,
     que son los que más se ausentan.
 
  5. INTERVENCIÓN FOCALIZADA EN VENTAS.
     Auditar las condiciones específicas del departamento (presión comercial,
     objetivos, jornada) en lugar de aplicar medidas genéricas a toda la empresa.
 
  6. SISTEMA DE ALERTA TEMPRANA, NO PUNITIVO.
     Combinar la caída de motivación/satisfacción con el aumento de horas
     extra como señal para OFRECER APOYO, con enfoque de cuidado, no de sanción.
""")

  APARTADO 3: ACCIONES PARA REDUCIR EL ABSENTISMO

Las medidas se ordenan según el peso de cada variable en el análisis:

  1. MEJORAR MOTIVACIÓN Y SATISFACCIÓN (prioridad máxima).
     Encuestas de clima periódicas con planes de acción por departamento,
     reconocimiento del trabajo y revisión del encaje puesto-persona.

  2. CONTROLAR LA SOBRECARGA Y LAS HORAS EXTRAS.
     Poner límites a las horas extra, redimensionar plantillas en las áreas
     más tensionadas y vigilar el agotamiento como señal de alerta temprana.

  3. REACTIVAR LA PROMOCIÓN INTERNA.
     Diseñar planes de carrera y revisar los casos de empleados con más de
     tres años sin promocionar (muy ligados al absentismo alto).

  4. REFORZAR LA FORMACIÓN.
     Ampliar el plan formativo, sobre todo en quienes reciben menos horas,
     que son los que más se ausentan.

  5. INTERVENCIÓN FOCALIZADA EN VENTAS.
     Auditar las condiciones específicas del departamento (presión comercial,
     objetivos, jornada) en lugar

In [7]:
#  GRÁFICOS (se guardan como imágenes PNG en la misma carpeta)
print("=" * 75)
print("  GENERANDO GRÁFICOS...")
print("=" * 75)
 
# --- Gráfico 1: correlación de cada variable con el absentismo ---
plt.figure(figsize=(8, 4.5))
colores = ["#dc2626" if r > 0 else "#16a34a" for r in valores_corr]
plt.barh(nombres_corr, valores_corr, color=colores)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlación de cada variable con el absentismo (Factor Bradford)")
plt.xlabel("Correlación de Pearson (rojo = sube, verde = baja)")
plt.tight_layout()
plt.savefig("grafico_1_correlaciones.png", dpi=130)
plt.close()
 
# --- Gráfico 2: absentismo medio por departamento ---
plt.figure(figsize=(7, 4.5))
medias_dept = df.groupby("Departamento")[BRADFORD].mean().sort_values()
plt.bar(medias_dept.index, medias_dept.values, color="#2563eb")
plt.title("Factor Bradford medio por departamento")
plt.ylabel("Factor Bradford medio")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("grafico_2_departamentos.png", dpi=130)
plt.close()
 
# --- Gráfico 3: perfil Bajo vs Alto ---
variables_perfil = ["Motivación (1-10)", "Satisfacción (1-10)",
                    "Horas Extras", "Meses desde última promoción",
                    "Horas de Formación (h)"]
medias_bajo = [grupo_bajo[v].mean() for v in variables_perfil]
medias_alto = [grupo_alto[v].mean() for v in variables_perfil]
posiciones = np.arange(len(variables_perfil))
ancho = 0.38
plt.figure(figsize=(9, 4.5))
plt.bar(posiciones - ancho/2, medias_bajo, ancho, label="Absentismo Bajo", color="#16a34a")
plt.bar(posiciones + ancho/2, medias_alto, ancho, label="Absentismo Alto", color="#dc2626")
plt.xticks(posiciones, ["Motivación", "Satisfacción", "Horas\nExtras",
                        "Meses sin\npromoción", "Formación"])
plt.title("Perfil del trabajador: absentismo Bajo vs Alto")
plt.legend()
plt.tight_layout()
plt.savefig("grafico_3_perfil.png", dpi=130)
plt.close()
 
# --- Gráfico 4: reparto de la plantilla por nivel ---
plt.figure(figsize=(6, 4.5))
niveles_orden = ["Bajo", "Medio", "Alto"]
cantidades = [conteo_niveles.get(n, 0) for n in niveles_orden]
plt.bar(niveles_orden, cantidades, color=["#16a34a", "#f59e0b", "#dc2626"])
plt.title("Plantilla por nivel de absentismo")
plt.ylabel("Nº de empleados")
plt.tight_layout()
plt.savefig("grafico_4_niveles.png", dpi=130)
plt.close()
 
print("Gráficos guardados:")
print("   grafico_1_correlaciones.png")
print("   grafico_2_departamentos.png")
print("   grafico_3_perfil.png")
print("   grafico_4_niveles.png")
print()
print("ANÁLISIS COMPLETADO.")

  GENERANDO GRÁFICOS...
Gráficos guardados:
   grafico_1_correlaciones.png
   grafico_2_departamentos.png
   grafico_3_perfil.png
   grafico_4_niveles.png

ANÁLISIS COMPLETADO.
